# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — Content Age and Decline

The paper reports that growing pages were younger on average than declining pages: 185 days versus 228 days. It also reports that word count was almost the same between the groups, so age was the clearer observed difference.

**Methodology question:** How was `trend_direction` defined for the growing and declining groups, and does the observation window ensure that the outcome is measured after the age-related features rather than at the same point in time?

Finding 2 — The Freshness Multiplier

The paper reports that freshness was associated with different growth-to-decline ratios across freshness windows. It also reports a separate comparison for pages older than a year, where recently refreshed pages showed higher health and impressions than pages last updated 181–360 days ago.

Methodology question:Because the refreshed and stale groups may differ in page quality, traffic, or other characteristics before the refresh, how does the analysis separate the observed association with refreshing from differences that existed before the refresh?

My review approach

These are constructive methodology questions rather than claims that the findings are wrong. I would want the label construction, timing, and comparison design to be clear before interpreting the observed relationships as evidence that age or refreshing itself causes the outcome.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

# Load data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Target
df = df.dropna(subset=["trend_direction"]).copy()
df["needs_attention"] = (
    df["trend_direction"] == "down"
).astype(int)

# Same features used in Week 5
numeric_features = [
    "search_volume",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "content_age_days",
    "ctr"
]

categorical_features = ["competition_level"]

features = numeric_features + categorical_features

X = df[features]
y = df["needs_attention"]

# Same preprocessing/model as W05
preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

def build_model():
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

# --------------------------------------------------
# BEFORE: ordinary random split
# --------------------------------------------------

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = build_model()
random_model.fit(X_train_random, y_train_random)

random_score = random_model.predict_proba(X_test_random)[:, 1]

random_ap = average_precision_score(
    y_test_random,
    random_score
)

random_auc = roc_auc_score(
    y_test_random,
    random_score
)

# --------------------------------------------------
# AFTER: grouped split by client
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=df["client_id"])
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

grouped_model = build_model()
grouped_model.fit(X_train_grouped, y_train_grouped)

grouped_score = grouped_model.predict_proba(X_test_grouped)[:, 1]

grouped_ap = average_precision_score(
    y_test_grouped,
    grouped_score
)

grouped_auc = roc_auc_score(
    y_test_grouped,
    grouped_score
)

# --------------------------------------------------
# Comparison
# --------------------------------------------------

comparison = pd.DataFrame({
    "validation_design": [
        "Random 80/20 split",
        "Grouped 80/20 split by client"
    ],
    "average_precision": [
        random_ap,
        grouped_ap
    ],
    "roc_auc": [
        random_auc,
        grouped_auc
    ]
})

print("Validation comparison:")
print(comparison.to_string(index=False))

# Verify grouped split
train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("\nGrouped split client overlap:",
      len(train_clients & test_clients))

Validation comparison:
            validation_design  average_precision  roc_auc
           Random 80/20 split           0.625136 0.611071
Grouped 80/20 split by client           0.558214 0.562052

Grouped split client overlap: 0


Before vs After

The random split gives an optimistic estimate because pages from the same client can appear in both training and test data.

The grouped split keeps each client's pages together, so the test set represents performance on clients not seen during training.

The grouped result is therefore the more honest estimate for this task. The comparison shows how much the validation design can change the measured model performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
# W05 final feature set
model_features = [
    "search_volume",
    "competition_level",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "content_age_days",
    "ctr"
]

# Explicit outcome / label fields
outcome_fields = [
    "trend_direction",
    "trend_pct"
]

print("FEATURE LEAKAGE AUDIT")
print("=" * 50)

print("\nFeatures used by W05:")
for feature in model_features:
    print("✓", feature)

print("\nOutcome fields:")
for field in outcome_fields:
    if field in df.columns:
        print("⚠", field, "-> NOT used as a model feature")

# Check whether any outcome field is accidentally in the feature list
leaked = set(model_features) & set(outcome_fields)

print("\nPotential label leakage:", leaked)

# Check for obvious future/outcome-related column names
future_keywords = [
    "future",
    "next",
    "outcome",
    "label",
    "target",
    "trend"
]

suspicious = [
    col for col in model_features
    if any(word in col.lower() for word in future_keywords)
]

print("Suspicious feature names:", suspicious)

if not leaked and not suspicious:
    print("\nLEAKAGE AUDIT: PASS")
else:
    print("\nLEAKAGE AUDIT: REVIEW NEEDED")

FEATURE LEAKAGE AUDIT

Features used by W05:
✓ search_volume
✓ competition_level
✓ cpc
✓ word_count
✓ char_count
✓ impressions_90d
✓ clicks_90d
✓ pageviews_90d
✓ sessions_90d
✓ content_age_days
✓ ctr

Outcome fields:
⚠ trend_direction -> NOT used as a model feature
⚠ trend_pct -> NOT used as a model feature

Potential label leakage: set()
Suspicious feature names: []

LEAKAGE AUDIT: PASS


Leakage Audit Result

The Week-5 model does not directly use `trend_direction` or `trend_pct` as features. These are outcome-related fields and are kept outside the feature set.

The selected features describe page, search, and recent performance characteristics rather than directly encoding the target.

I also checked the feature names for obvious future-window or target-derived fields. No such feature was intentionally included.

This audit does not prove that every feature is perfectly causal or temporally isolated; it establishes that the model feature list does not directly contain the known outcome fields.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [6]:
print("Original claim:")
print(
    "The Logistic Regression model accurately predicts which content "
    "pages need to be refreshed and performs better than the baseline."
)

print("\nRewritten claim:")
print(
    "On the evaluated test split, Logistic Regression measured higher "
    "Average Precision and ROC-AUC than the Week-4 rule-based baseline. "
    "This provides directional evidence that the model may improve "
    "content-review prioritization, but it does not prove that refreshing "
    "the recommended pages will improve future search performance."
)

Original claim:
The Logistic Regression model accurately predicts which content pages need to be refreshed and performs better than the baseline.

Rewritten claim:
On the evaluated test split, Logistic Regression measured higher Average Precision and ROC-AUC than the Week-4 rule-based baseline. This provides directional evidence that the model may improve content-review prioritization, but it does not prove that refreshing the recommended pages will improve future search performance.


Why I changed the claim

The original wording was too strong because the evaluation measures model ranking performance, not whether a refresh actually improves a page.

The measured results support saying that Logistic Regression performed better than the baseline on the evaluated split. They do not establish causality or guarantee that the recommended pages will benefit from a refresh.

Therefore, I describe the result as observed, measured, directional, and useful for decision-support.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.